In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
from scripts.plotting import *
from scripts.denoising import *
from scripts.TPS import *

In [ ]:
X = np.array(pd.read_csv("./data/s_curve/uniform/s_curve_noisy_position_matrix.csv"))
Y = np.array(pd.read_csv("./data/s_curve/uniform/s_curve_noisy_velocity_matrix.csv"))
t = pd.read_csv("./data/s_curve/uniform/s_curve_gt_latent_time_vector.csv")
t = list(t["t"])

X_gt = np.array(pd.read_csv("./data/s_curve/uniform/s_curve_gt_position_matrix.csv"))
Y_gt = np.array(pd.read_csv("./data/s_curve/uniform/s_curve_gt_velocity_matrix.csv"))

# X = X_gt
# Y = Y_gt

X.shape, Y.shape

In [ ]:
# Visualize using the provided function
plot_3d_with_quiver(
    X,
    Y,
    t,
    arrow_size=0.2
)

In [ ]:
import umap

umap_reducer = umap.UMAP(n_neighbors=15, min_dist=0.5, n_components=2, random_state=42)
X_2d = umap_reducer.fit_transform(X)
plot_2d(X_2d, t, "UMAP")

In [ ]:
tps = ThinPlateSpline(X_2d, n_control_points=1000)
tps.fit(X, dof_target=50)
metrics = tps.evaluate_fit(X)
metrics

In [ ]:
X_smoothed = tps.predict(X_2d)
plot_3d(X_smoothed,t)

In [ ]:
import numpy as np

# Define grid resolution
num_grid_points = 30  # Adjust based on desired granularity

# Compute the min and max of X_2d
x_min, y_min = X_2d.min(axis=0)
x_max, y_max = X_2d.max(axis=0)

# Generate grid points
x_lin = np.linspace(x_min, x_max, num_grid_points)
y_lin = np.linspace(y_min, y_max, num_grid_points)
grid_x, grid_y = np.meshgrid(x_lin, y_lin)  # Create a 2D grid
grid_points = np.column_stack([grid_x.ravel(), grid_y.ravel()])  # Flatten grid

# Confirm it's not affecting TPS fitting
plot_2d(X_2d, t, "UMAP with Grid Points")
plot_2d(grid_points, np.zeros(grid_points.shape[0]), "Grid Points")

In [ ]:
tps = ThinPlateSpline(X_2d, n_control_points=num_grid_points**2)
tps.control_points = grid_points
pairwise_distances = cdist(X_2d, tps.control_points, metric="euclidean")
tps.K = tps.tps_kernel(pairwise_distances)
tps.fit(X, dof_target=30)

In [ ]:
X_smoothed = tps.predict(X_2d)
plot_3d(X_smoothed,t)

In [ ]:
import plotly.graph_objects as go
import numpy as np

grid_3d = tps.predict(grid_points)
# Stack grid points and data points together
combined_3d = np.vstack([grid_3d, X_smoothed])
combined_colors = [0] * len(grid_3d) + list(t)  # Grey for grid, t for data

# Create an interactive scatter plot
fig = go.Figure()

# Add grid points (grey)
fig.add_trace(go.Scatter3d(
    x=grid_3d[:, 0], y=grid_3d[:, 1], z=grid_3d[:, 2],
    mode='markers',
    marker=dict(size=3, color='grey'),
    name='Grid Points'
))

# Add data points (colored by t)
fig.add_trace(go.Scatter3d(
    x=X_smoothed[:, 0], y=X_smoothed[:, 1], z=X_smoothed[:, 2],
    mode='markers',
    marker=dict(size=3, color=t, colorscale='Viridis', opacity=0.8),
    name='Data Points'
))

# Layout settings
fig.update_layout(
    title="Interactive 3D Plot of Grid and Data Points",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z"
    )
)

# Show interactive plot
fig.show()


In [ ]:
jacobians = tps.compute_tps_jacobians(X_2d)

projected_velocities = np.zeros_like(X_2d)
num_points = X_2d.shape[0]
for i in range(num_points):
    projected_velocities[i], _, _, _ = np.linalg.lstsq(jacobians[i], Y[i], rcond=None)


tps_vf = ThinPlateSpline(X_2d, n_control_points=num_grid_points**2)
tps_vf.control_points = grid_points
pairwise_distances = cdist(X_2d, tps_vf.control_points, metric="euclidean")
tps_vf.K = tps_vf.tps_kernel(pairwise_distances)
tps_vf.fit(projected_velocities, dof_target=20)    
smoothed_velocities = tps_vf.predict(X_2d) 

plot_2d_quiver(X_2d, smoothed_velocities, t, scale=5, cmap='coolwarm', arrow_color='black')

In [ ]:
from scipy.optimize import minimize

In [ ]:
# Total loss function: L_total = L1 + λ * L2
def total_loss(X_flat, tps, tps_vf, X, Y, lam):
    # Reshape the 1D array into 2D points (n, 2)
    X_2d = X_flat.reshape((-1, 2))
    jacobians_f = tps.compute_tps_jacobians(X_2d)
    # L1: Compare original X with TPS prediction from tps
    X_smoothed = tps.predict(X_2d)
    loss1 = np.sum((X - X_smoothed) ** 2)
    
    # L2: Compare Y with TPS prediction from tps_vf
    vf_smoothed = tps_vf.predict(X_2d)
    Y_smoothed = np.einsum("naj, nj -> na", jacobians_f, vf_smoothed, optimize=True)
    loss2 = np.sum((Y - Y_smoothed) ** 2)
    return loss1 + lam * loss2

# Total gradient: gradient of L_total = grad(L1) + λ * grad(L2)
def total_gradient(X_flat, tps, tps_vf, X, Y, lam):
    # Reshape the 1D array back into 2D points
    X_2d = X_flat.reshape((-1, 2))
    
    # Gradient for L1
    X_smoothed = tps.predict(X_2d)
    jacobians_f = tps.compute_tps_jacobians(X_2d)
    hessian_f = tps.compute_tps_hessians(X_2d)
    grad1 = -2 * np.einsum("nai,na -> ni", jacobians_f, (X - X_smoothed), optimize=True)
    
    # Gradient for L2
    vf_smoothed = tps_vf.predict(X_2d)
    jacobians_v = tps_vf.compute_tps_jacobians(X_2d)
    Y_smoothed = np.einsum("naj, nj -> na", jacobians_f, vf_smoothed, optimize=True)
    A = -2*(Y - Y_smoothed)
    B = np.einsum("naij, nj -> nai", hessian_f, vf_smoothed, optimize=True)
    C = np.einsum("naj, nji -> nai", jacobians_f, jacobians_v, optimize=True)
    grad2 = np.einsum("na, nai -> ni", A, (B+C), optimize=True)
    # Total gradient is the sum of the two (with L2 weighted by lam)
    total_grad = grad1 + lam * grad2
    
    # Return as a flattened array (since the optimizer expects 1D arrays)
    return total_grad.flatten()

# Optimization function that minimizes the total loss
def optimize_total(tps, tps_vf, X_2d, X, Y, lam, disp=False):
    # X here is your initial guess for the 2D points, of shape (n, 2)
    X_flat = X_2d.flatten()  # Flatten for the optimizer
    result = minimize(
        fun=total_loss,
        x0=X_flat,
        jac=total_gradient,
        args=(tps, tps_vf, X, Y, lam),
        method='L-BFGS-B',
        options={"disp": disp}
    )
    # Reshape the optimized variable back into (n, 2)
    X_optimized = result.x.reshape((-1, 2))
    return X_optimized, result

# Example usage:
# Suppose tps and tps_vf are your TPS objects,
# X is your original set of points (n, 2),
# Y is the target set for the second term (n, 2),
# and lam is your weighting parameter.
#
lam = 1
%time X_optimized, optimization_result = optimize_total(tps, tps_vf, X_2d, X, Y, lam, disp=True)

In [ ]:
N = X.shape[0]
k = 10
sigma = 1

# Compute pairwise distances in high-dimensional space
D_high = cdist(X, X, metric='sqeuclidean')  # (N, N) squared Euclidean distance

# Find k-nearest neighbors (excluding self)
neighbors = np.argsort(D_high, axis=1)[:, 1:k+1]  # (N, k)

# Construct weight matrix using Gaussian kernel
W = np.exp(-D_high / (2 * sigma**2))  # (N, N)

# Extract relevant weights (only for k-NN)
row_idx = np.repeat(np.arange(N), k)
col_idx = neighbors.ravel()
W_values = W[row_idx, col_idx]  # (N*k,)

# Compute pairwise differences in low-dimensional space
X2D_diff = X_2d[row_idx] - X_2d[col_idx]  # (N*k, 2)

# Compute smoothness loss
L_smooth = np.sum(W_values * np.sum(X2D_diff**2, axis=1))  # Scalar

# Compute gradient
grad = np.zeros_like(X_2d)
np.add.at(grad, row_idx, 2 * W_values[:, None] * X2D_diff)
np.add.at(grad, col_idx, -2 * W_values[:, None] * X2D_diff)  # Symmetric update
grad

In [ ]:
import numpy as np
from scipy.optimize import minimize

# Global cache to store previously computed loss and gradient
_loss_cache = None
_grad_cache = None

def compute_loss_and_gradient(X_flat, tps, tps_vf, X, Y, lam, mu=0.1, k=10, sigma=1.0):
    """Computes both the function value and gradient, including graph Laplacian smoothness."""
    X_2d = X_flat.reshape((-1, 2))

    # Compute shared values
    X_smoothed = tps.predict(X_2d)
    jacobians_f = tps.compute_tps_jacobians(X_2d)
    hessian_f = tps.compute_tps_hessians(X_2d)

    # Loss L1 (Reconstruction loss)
    loss1 = np.sum((X - X_smoothed) ** 2)

    # Loss L2 (Velocity field constraint)
    vf_smoothed = tps_vf.predict(X_2d)
    Y_smoothed = np.einsum("naj, nj -> na", jacobians_f, vf_smoothed)
    loss2 = np.sum((Y - Y_smoothed) ** 2)
    
    # Loss L3 (Graph Laplacian smoothness)
    D_high = cdist(X, X, metric='sqeuclidean')  # High-dimensional pairwise distances
    neighbors = np.argsort(D_high, axis=1)[:, 1:k+1]  # k-NN indices

    W = np.exp(-D_high / (2 * sigma**2))  # Weight matrix
    row_idx = np.repeat(np.arange(X.shape[0]), k)
    col_idx = neighbors.ravel()
    W_values = W[row_idx, col_idx]  # Only for k-NN pairs

    X2D_diff = X_2d[row_idx] - X_2d[col_idx]  # Differences in 2D space
    loss3 = np.sum(W_values * np.sum(X2D_diff**2, axis=1))  # Scalar loss

    # Compute Gradients
    grad1 = -2 * np.einsum("nai,na -> ni", jacobians_f, (X - X_smoothed))

    A = -2 * (Y - Y_smoothed)
    B = np.einsum("naij, nj -> nai", hessian_f, vf_smoothed)
    C = np.einsum("naj, nji -> nai", jacobians_f, tps_vf.compute_tps_jacobians(X_2d))
    grad2 = np.einsum("na, nai -> ni", A, (B + C))
    
    # Compute Gradient of L3
    grad3 = np.zeros_like(X_2d)
    np.add.at(grad3, row_idx, 2 * W_values[:, None] * X2D_diff)
    np.add.at(grad3, col_idx, -2 * W_values[:, None] * X2D_diff)  # Symmetric update

    # Total Loss and Gradient
    total_loss_value = loss1 + lam * loss2 + mu * loss3
    total_grad_value = (grad1 + lam * grad2 + mu * grad3).flatten()

    return total_loss_value, total_grad_value

def total_loss(X_flat, tps, tps_vf, X, Y, lam):
    """Wrapper for computing and caching the loss function."""
    global _loss_cache, _grad_cache
    
    # Check cache
    if _loss_cache is not None and np.allclose(X_flat, _loss_cache[0]):
        return _loss_cache[1]

    # Compute function and gradient together
    loss_value, grad_value = compute_loss_and_gradient(X_flat, tps, tps_vf, X, Y, lam)

    # Cache results
    _loss_cache = (X_flat.copy(), loss_value)
    _grad_cache = (X_flat.copy(), grad_value)

    return loss_value

def total_gradient(X_flat, tps, tps_vf, X, Y, lam):
    """Wrapper for computing and caching the gradient."""
    global _grad_cache

    # Check cache
    if _grad_cache is not None and np.allclose(X_flat, _grad_cache[0]):
        return _grad_cache[1]

    # Compute function and gradient together
    loss_value, grad_value = compute_loss_and_gradient(X_flat, tps, tps_vf, X, Y, lam)

    # Cache results
    _loss_cache = (X_flat.copy(), loss_value)
    _grad_cache = (X_flat.copy(), grad_value)

    return grad_value


def optimize_total(tps, tps_vf, X_2d, X, Y, lam, method="L-BFGS-B", tol=1e-4, disp=False):
    """
    Optimizes the total loss function using L-BFGS-B (or other methods).
    
    Args:
        tps: TPS model for X smoothing.
        tps_vf: TPS model for vector field.
        X_2d: Initial 2D points (n, 2).
        X: Original high-dimensional data points.
        Y: Vector field data at each point.
        lam: Regularization parameter.
        method: Optimization method (default: 'L-BFGS-B').
        tol: Tolerance for stopping criterion (higher means faster but less precise).
    
    Returns:
        X_optimized: Optimized 2D coordinates.
        result: SciPy optimization result object.
    """
    global _loss_cache, _grad_cache
    _loss_cache, _grad_cache = None, None  # Reset cache before optimization

    X_flat = X_2d.flatten()  # Flatten input for optimizer

    result = minimize(
        fun=total_loss,
        x0=X_flat,
        jac=total_gradient,
        args=(tps, tps_vf, X, Y, lam),
        method=method,
        tol=tol,  # Set higher tolerance
        options={
            "gtol": tol,   # Gradient norm stopping criterion
            "ftol": tol,   # Function value stopping criterion
            "disp": disp   # Display optimization progress
        }
    )
    X_optimized = result.x.reshape((-1, 2))  # Reshape back to 2D
    return X_optimized, result


lam = 1
mu = 1
%time X_optimized, optimization_result = optimize_total(tps, tps_vf, X_2d, X, Y, lam, disp=True)

In [ ]:
plot_2d(X_2d, t)
plot_2d(X_optimized, t)

In [ ]:
# Step 1: Initialize with PCA
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X)  # Initial embedding

# Define grid resolution
num_grid_points = 30  # Adjust based on desired granularity

# Compute the min and max of X_2d
x_min, y_min = X_2d.min(axis=0)
x_max, y_max = X_2d.max(axis=0)

# Generate grid points
x_lin = np.linspace(x_min, x_max, num_grid_points)
y_lin = np.linspace(y_min, y_max, num_grid_points)
grid_x, grid_y = np.meshgrid(x_lin, y_lin)  # Create a 2D grid
grid_points = np.column_stack([grid_x.ravel(), grid_y.ravel()])  # Flatten grid

# Confirm it's not affecting TPS fitting
plot_2d(X_2d, t)
# plot_2d(grid_points, np.zeros(grid_points.shape[0]), "Grid Points")

lam = 2
num_iterations = 3
for i in range(num_iterations):
    print(f"Iteration {i+1}")

    # Step 2: Fit the thin-plate spline (TPS)
    tps = ThinPlateSpline(X_2d, n_control_points=num_grid_points**2)
    tps.control_points = grid_points
    pairwise_distances = cdist(X_2d, tps.control_points, metric="euclidean")
    tps.K = tps.tps_kernel(pairwise_distances)
    tps.fit(X, lambda_reg=3e5)
    
    jacobians = tps.compute_tps_jacobians(X_2d)
    projected_velocities = np.zeros_like(X_2d)
    num_points = X_2d.shape[0]
    for i in range(num_points):
        projected_velocities[i], _, _, _ = np.linalg.lstsq(jacobians[i], Y[i], rcond=None)

    tps_vf = ThinPlateSpline(X_2d, n_control_points=num_grid_points**2)
    tps_vf.control_points = grid_points
    pairwise_distances = cdist(X_2d, tps_vf.control_points, metric="euclidean")
    tps_vf.K = tps_vf.tps_kernel(pairwise_distances)
    tps_vf.fit(projected_velocities, lambda_reg=3e5)
    
    X_2d, optimization_result = optimize_total(tps, tps_vf, X_2d, X, Y, lam)

    # Final visualization
    plot_2d(X_2d, t, "Iterative TPS Optimization")
    vf_smoothed = tps_vf.predict(X_2d) 
    plot_2d_quiver(X_2d, vf_smoothed, t, scale=10, cmap='coolwarm', arrow_color='black')

In [ ]:
# Step 1: Initialize with PCA
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X)  # Initial embedding

# Confirm it's not affecting TPS fitting
plot_2d(X_2d, t)
# plot_2d(grid_points, np.zeros(grid_points.shape[0]), "Grid Points")

lam = 2
num_iterations = 3
for i in range(num_iterations):
    print(f"Iteration {i+1}")

    # Step 2: Fit the thin-plate spline (TPS)
    tps = ThinPlateSpline(X_2d)
    tps.fit(X, lambda_reg=3e5)
    
    jacobians = tps.compute_tps_jacobians(X_2d)
    projected_velocities = np.zeros_like(X_2d)
    num_points = X_2d.shape[0]
    for i in range(num_points):
        projected_velocities[i], _, _, _ = np.linalg.lstsq(jacobians[i], Y[i], rcond=None)

    tps_vf = ThinPlateSpline(X_2d)
    tps_vf.fit(projected_velocities, lambda_reg=3e5)
    
    X_2d, optimization_result = optimize_total(tps, tps_vf, X_2d, X, Y, lam)

    # Final visualization
    plot_2d(X_2d, t, "Iterative TPS Optimization")
    vf_smoothed = tps_vf.predict(X_2d) 
    plot_2d_quiver(X_2d, vf_smoothed, t, scale=10, cmap='coolwarm', arrow_color='black')

In [ ]:
# Step 1: Initialize with umap
umap_reducer = umap.UMAP(n_neighbors=15, min_dist=0.5, n_components=2, random_state=42)
X_2d = umap_reducer.fit_transform(X)

# Define grid resolution
num_grid_points = 30  # Adjust based on desired granularity

# Compute the min and max of X_2d
x_min, y_min = X_2d.min(axis=0)
x_max, y_max = X_2d.max(axis=0)

# Generate grid points
x_lin = np.linspace(x_min, x_max, num_grid_points)
y_lin = np.linspace(y_min, y_max, num_grid_points)
grid_x, grid_y = np.meshgrid(x_lin, y_lin)  # Create a 2D grid
grid_points = np.column_stack([grid_x.ravel(), grid_y.ravel()])  # Flatten grid

# Confirm it's not affecting TPS fitting
plot_2d(X_2d, t)
# plot_2d(grid_points, np.zeros(grid_points.shape[0]), "Grid Points")

lam = 1
num_iterations = 3
for i in range(num_iterations):
    print(f"Iteration {i+1}")

    # Step 2: Fit the thin-plate spline (TPS)
    tps = ThinPlateSpline(X_2d, n_control_points=num_grid_points**2)
    tps.control_points = grid_points
    pairwise_distances = cdist(X_2d, tps.control_points, metric="euclidean")
    tps.K = tps.tps_kernel(pairwise_distances)
    tps.fit(X, lambda_reg=3e5)
    
    jacobians = tps.compute_tps_jacobians(X_2d)
    projected_velocities = np.zeros_like(X_2d)
    num_points = X_2d.shape[0]
    for i in range(num_points):
        projected_velocities[i], _, _, _ = np.linalg.lstsq(jacobians[i], Y[i], rcond=None)

    tps_vf = ThinPlateSpline(X_2d, n_control_points=num_grid_points**2)
    tps_vf.control_points = grid_points
    pairwise_distances = cdist(X_2d, tps_vf.control_points, metric="euclidean")
    tps_vf.K = tps_vf.tps_kernel(pairwise_distances)
    tps_vf.fit(projected_velocities, lambda_reg=3e5)
    
    X_2d, optimization_result = optimize_total(tps, tps_vf, X_2d, X, Y, lam)

    # Final visualization
    plot_2d(X_2d, t, "Iterative TPS Optimization")
    vf_smoothed = tps_vf.predict(X_2d) 
    plot_2d_quiver(X_2d, vf_smoothed, t, scale=10, cmap='coolwarm', arrow_color='black')

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, perplexity=30, learning_rate=200, random_state=42)
X_2d = tsne.fit_transform(X)

lam = 1
num_iterations = 3
for i in range(num_iterations):
    print(f"Iteration {i+1}")

    # Step 2: Fit the thin-plate spline (TPS)
    tps = ThinPlateSpline(X_2d)
    tps.fit(X, dof_target=50)
    
    jacobians = tps.compute_tps_jacobians(X_2d)
    projected_velocities = np.zeros_like(X_2d)
    num_points = X_2d.shape[0]
    for i in range(num_points):
        projected_velocities[i], _, _, _ = np.linalg.lstsq(jacobians[i], Y[i], rcond=None)

    tps_vf = ThinPlateSpline(X_2d)
    tps_vf.fit(projected_velocities, dof_target=50)
    
    X_2d, optimization_result = optimize_total(tps, tps_vf, X_2d, X, Y, lam)

    # Final visualization
    plot_2d(X_2d, t, "Iterative TPS Optimization")
    vf_smoothed = tps_vf.predict(X_2d) 
    plot_2d_quiver(X_2d, vf_smoothed, t, scale=10, cmap='coolwarm', arrow_color='black')

In [ ]:
from sklearn.manifold import SpectralEmbedding

spectral = SpectralEmbedding(n_components=2, n_neighbors=10, random_state=42)
X_2d = spectral.fit_transform(X)

# Confirm it's not affecting TPS fitting
plot_2d(X_2d, t)
lam = 1
num_iterations = 3
for i in range(num_iterations):
    print(f"Iteration {i+1}")

    # Step 2: Fit the thin-plate spline (TPS)
    tps = ThinPlateSpline(X_2d)
    tps.fit(X, lambda_reg=5)
    
    jacobians = tps.compute_tps_jacobians(X_2d)
    projected_velocities = np.zeros_like(X_2d)
    num_points = X_2d.shape[0]
    for i in range(num_points):
        projected_velocities[i], _, _, _ = np.linalg.lstsq(jacobians[i], Y[i], rcond=None)

    tps_vf = ThinPlateSpline(X_2d)
    tps_vf.fit(projected_velocities, lambda_reg=2000)
    
    X_2d, optimization_result = optimize_total(tps, tps_vf, X_2d, X, Y, lam)

    # Final visualization
    plot_2d(X_2d, t, "Iterative TPS Optimization")
    vf_smoothed = tps_vf.predict(X_2d) 
    plot_2d_quiver(X_2d, vf_smoothed, t, scale=10, cmap='coolwarm', arrow_color='black')

In [ ]:
from sklearn.manifold import Isomap

isomap = Isomap(n_components=2, n_neighbors=10)
X_2d = isomap.fit_transform(X)

# Define grid resolution
num_grid_points = 30  # Adjust based on desired granularity

# Compute the min and max of X_2d
x_min, y_min = X_2d.min(axis=0)
x_max, y_max = X_2d.max(axis=0)

# Generate grid points
x_lin = np.linspace(x_min, x_max, num_grid_points)
y_lin = np.linspace(y_min, y_max, num_grid_points)
grid_x, grid_y = np.meshgrid(x_lin, y_lin)  # Create a 2D grid
grid_points = np.column_stack([grid_x.ravel(), grid_y.ravel()])  # Flatten grid

# Confirm it's not affecting TPS fitting
plot_2d(X_2d, t)

lam = 1
num_iterations = 3
for i in range(num_iterations):
    print(f"Iteration {i+1}")

    # Step 2: Fit the thin-plate spline (TPS)
    tps = ThinPlateSpline(X_2d, n_control_points=num_grid_points**2)
    tps.control_points = grid_points
    pairwise_distances = cdist(X_2d, tps.control_points, metric="euclidean")
    tps.K = tps.tps_kernel(pairwise_distances)
    tps.fit(X, dof_target=30)
    
    jacobians = tps.compute_tps_jacobians(X_2d)
    projected_velocities = np.zeros_like(X_2d)
    num_points = X_2d.shape[0]
    for i in range(num_points):
        projected_velocities[i], _, _, _ = np.linalg.lstsq(jacobians[i], Y[i], rcond=None)

    tps_vf = ThinPlateSpline(X_2d, n_control_points=num_grid_points**2)
    tps_vf.control_points = grid_points
    pairwise_distances = cdist(X_2d, tps_vf.control_points, metric="euclidean")
    tps_vf.K = tps_vf.tps_kernel(pairwise_distances)
    tps_vf.fit(projected_velocities, dof_target=30)
    
    X_2d, optimization_result = optimize_total(tps, tps_vf, X_2d, X, Y, lam)

    # Final visualization
    plot_2d(X_2d, t, "Iterative TPS Optimization")
    vf_smoothed = tps_vf.predict(X_2d) 
    plot_2d_quiver(X_2d, vf_smoothed, t, scale=10, cmap='coolwarm', arrow_color='black')

In [ ]:
from sklearn.manifold import Isomap


# Example usage:
X_2d = np.random.uniform(0, 1, size=(1000, 2))


# Confirm it's not affecting TPS fitting
plot_2d(X_2d, t)

lam = 1
num_iterations = 3
for i in range(num_iterations):
    print(f"Iteration {i+1}")

    # Step 2: Fit the thin-plate spline (TPS)
    tps = ThinPlateSpline(X_2d)
    tps.fit(X, dof_target=30)
    
    jacobians = tps.compute_tps_jacobians(X_2d)
    projected_velocities = np.zeros_like(X_2d)
    num_points = X_2d.shape[0]
    for i in range(num_points):
        projected_velocities[i], _, _, _ = np.linalg.lstsq(jacobians[i], Y[i], rcond=None)

    tps_vf = ThinPlateSpline(X_2d)
    tps_vf.fit(projected_velocities, dof_target=30)
    
    X_2d, optimization_result = optimize_total(tps, tps_vf, X_2d, X, Y, lam)

    # Final visualization
    plot_2d(X_2d, t, "Iterative TPS Optimization")
    vf_smoothed = tps_vf.predict(X_2d) 
    plot_2d_quiver(X_2d, vf_smoothed, t, scale=10, cmap='coolwarm', arrow_color='black')